In [0]:
dbutils.widgets.text("catalog",        "banking")
dbutils.widgets.text("schema_landing", "landing")
dbutils.widgets.text("schema_bronze",  "bronze")

catalog        = dbutils.widgets.get("catalog")
schema_landing = dbutils.widgets.get("schema_landing")
schema_bronze  = dbutils.widgets.get("schema_bronze")


table_transactions     = f"{catalog}.{schema_landing}.transactions"
monitoring_table   = f"{catalog}.{schema_bronze}.execution_monitoring"
target_table = f"{catalog}.{schema_bronze}.bronze_transactions"


In [0]:
from pyspark.sql.functions import current_timestamp
import time
import uuid
from datetime import datetime

run_id = str(uuid.uuid4())
notebook_name = "06_ingestion_transactions"

start_time = datetime.now()

try:
   
    
    df_transactions = spark.read.table(table_transactions)
    df_transactions = df_transactions.withColumn('ingestion_date', current_timestamp())
    
    
    rows_processed = df_transactions.count()
    

    df_transactions.write.mode("overwrite").option("delta.feature.allowColumnDefaults", "supported").saveAsTable(target_table)
    
   
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    
    spark.sql(f"""
        INSERT INTO {monitoring_table}
        VALUES (
            '{run_id}',
            '{notebook_name}',
            '{target_table}',
            timestamp'{start_time.strftime('%Y-%m-%d %H:%M:%S')}',
            timestamp'{end_time.strftime('%Y-%m-%d %H:%M:%S')}',
            {duration},
            'SUCCESS',
            NULL,
            {rows_processed}
        )
    """)
    
    print(f"✗ Execution succefully")
    
except Exception as e:
    
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    error_message = str(e).replace("'", "''")
    
    spark.sql(f"""
        INSERT INTO {monitoring_table}
        VALUES (
            '{run_id}',
            '{notebook_name}',
            '{target_table}',
            timestamp'{start_time.strftime('%Y-%m-%d %H:%M:%S')}',
            timestamp'{end_time.strftime('%Y-%m-%d %H:%M:%S')}',
            {duration},
            'FAILED',
            '{error_message}',
            NULL
        )
    """)
    
    print(f"✗ Execution failed")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Error: {error_message}")
    
    raise